# 02 · Model selection — train it yourself

*Runnable notebook. We fit every model here and draw every figure from the data; nothing
is a pre-baked image. Set the kernel to **Python (coastal-et)** and Run All (~1–3 min).*

We ask one question three ways: given satellite indices + meteorology on a cloud-free
overpass, can we predict the tower's measured daily ET? The three validation schemes are
progressively harder — predicting at a **monitored** site (K-fold), in an **unseen year**
(leave-year), and at a **completely unseen tower** (leave-site, i.e. true upscaling).

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Derive the project root without hardcoding anyone's personal path:
#   env override -> parent of the notebook's folder -> shared fallback.
ROOT = os.environ.get("COASTAL_ET_ROOT")
if not ROOT or not os.path.isdir(os.path.join(ROOT, "data", "processed")):
    cand = os.path.dirname(os.getcwd())                 # notebook lives in <ROOT>/notebooks
    ROOT = cand if os.path.isdir(os.path.join(cand, "data", "processed")) \
        else "/anvil/projects/x-ees260113/team2/coastal-et"
PROC = f"{ROOT}/data/processed"
FIG = f"{ROOT}/figures"; os.makedirs(FIG, exist_ok=True)
print("project root:", ROOT)

# Nature-ish figure defaults
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 8, "axes.linewidth": 0.7, "figure.dpi": 120,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7})
INK = "#1a1a1a"

## Load the data

One self-contained table: 833 overpass matches across 13 coastal-wetland towers.

In [ ]:
# The 14 predictors: 7 satellite + 7 meteorology. Target is measured closed ET.
SAT = ["LAI", "EVI2", "SAVI", "NDVI", "NDWI", "MNDWI", "LST_K"]
MET = ["TA_ERA", "VPD_ERA", "SW_IN_ERA", "WS_ERA", "ETo_mm", "DOY_sin", "DOY_cos"]
FEATS = SAT + MET
EVERGLADES = ["US-Esm", "US-TaS", "US-Skr", "US-Elm", "US-EvM"]

d = pd.read_parquet(f"{PROC}/more_sites_table.parquet")
SITES = sorted(d.SITE_ID.unique())
print(f"{len(d)} overpass matches | {len(SITES)} sites | {len(FEATS)} features")
print("target: ET_closed_mm (measured, closure-corrected daily ET)")
d[["SITE_ID", "year"] + FEATS + ["ET_closed_mm"]].head()

## Cross-validation, honestly

The evaluator refits the model for every fold/year/site so nothing leaks across the
split we care about.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_absolute_error

def evaluate(make_model, data, scheme, feats=FEATS):
    """Return (R2, MAE, y_true, y_pred) for one CV scheme.
    kfold=predict at monitored sites; year=predict unseen years;
    site=predict a completely unseen tower (spatial upscaling)."""
    yt, yp = [], []
    if scheme == "kfold":
        for tri, tei in KFold(10, shuffle=True, random_state=0).split(data):
            m = clone(make_model()).fit(data.iloc[tri][feats].values, data.iloc[tri].ET_closed_mm.values)
            yp.append(m.predict(data.iloc[tei][feats].values)); yt.append(data.iloc[tei].ET_closed_mm.values)
    elif scheme == "year":
        for s in sorted(data.SITE_ID.unique()):
            ds = data[data.SITE_ID == s]
            for y in sorted(ds.year.dropna().unique()):
                tr, te = ds[ds.year != y], ds[ds.year == y]
                if len(te) < 5 or len(tr) < 15: continue
                m = clone(make_model()).fit(tr[feats].values, tr.ET_closed_mm.values)
                yp.append(m.predict(te[feats].values)); yt.append(te.ET_closed_mm.values)
    else:  # leave-site-out
        for s in sorted(data.SITE_ID.unique()):
            tr, te = data[data.SITE_ID != s], data[data.SITE_ID == s]
            if len(te) < 5: continue
            m = clone(make_model()).fit(tr[feats].values, tr.ET_closed_mm.values)
            yp.append(m.predict(te[feats].values)); yt.append(te.ET_closed_mm.values)
    yt, yp = np.concatenate(yt), np.concatenate(yp)
    return r2_score(yt, yp), mean_absolute_error(yt, yp), yt, yp

print("CV evaluator ready (schemes: kfold, year, site)")

## The model zoo

Linear, kernel, tree-ensemble, Gaussian-process and (if installed) boosted-tree models.

In [ ]:
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor)
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def model_zoo():
    z = {
        "Ridge":       lambda: make_pipeline(StandardScaler(), Ridge(alpha=10)),
        "ElasticNet":  lambda: make_pipeline(StandardScaler(), ElasticNet(alpha=0.05, l1_ratio=0.3, max_iter=5000)),
        "PLS":         lambda: make_pipeline(StandardScaler(), PLSRegression(n_components=6)),
        "kNN":         lambda: make_pipeline(StandardScaler(), KNeighborsRegressor(10, weights="distance")),
        "SVR":         lambda: make_pipeline(StandardScaler(), SVR(C=5, gamma="scale", epsilon=0.2)),
        "GaussProc":   lambda: make_pipeline(StandardScaler(), GaussianProcessRegressor(
                            kernel=ConstantKernel() * RBF() + WhiteKernel(), normalize_y=True, alpha=1e-3, random_state=0)),
        "RandomForest": lambda: RandomForestRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1),
        "ExtraTrees":  lambda: ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1),
        "GradBoost":   lambda: GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, random_state=0),
        "HistGBM":     lambda: HistGradientBoostingRegressor(max_iter=400, l2_regularization=1, random_state=0),
    }
    try:
        from xgboost import XGBRegressor
        z["XGBoost"] = lambda: XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.03,
                                            subsample=0.8, colsample_bytree=0.8, random_state=0, verbosity=0)
    except Exception: pass
    try:
        from lightgbm import LGBMRegressor
        z["LightGBM"] = lambda: LGBMRegressor(n_estimators=400, num_leaves=31, learning_rate=0.03,
                                              subsample=0.8, random_state=0, verbose=-1)
    except Exception: pass
    return z

ZOO = model_zoo()
print("model zoo:", list(ZOO.keys()))

## Train everything

This is the actual training run — each model is fit across all three schemes.

In [ ]:
# Train every model across all three schemes. This actually fits the models now
# (~1-3 min). Leave-site-out is the headline test: can we predict an UNSEEN tower?
rows = []
for name, mk in ZOO.items():
    rk, _, _, _ = evaluate(mk, d, "kfold")
    ry, _, _, _ = evaluate(mk, d, "year")
    rs, _, _, _ = evaluate(mk, d, "site")
    rows.append((name, rk, ry, rs))
    print(f"  {name:<13} kfold={rk:5.2f}  leave-year={ry:5.2f}  leave-site={rs:5.2f}", flush=True)

comp = pd.DataFrame(rows, columns=["model", "kfold", "leave_year", "leave_site"]).sort_values(
    "leave_site", ascending=False).reset_index(drop=True)
print(f"\nbest upscaler (leave-site): {comp.iloc[0].model}  R2={comp.iloc[0].leave_site:.2f}")
comp

## Visualize the comparison

In [ ]:
# Figure: R2 by validation scheme for the top models (draw it here, save a copy)
top = comp.head(6)
x = np.arange(len(top)); w = 0.26
fig, ax = plt.subplots(figsize=(6.4, 3.6), constrained_layout=True)
for i, (col, lab, c) in enumerate([("kfold", "K-fold (monitored)", "#B0C4DE"),
                                   ("leave_year", "leave-year (unseen years)", "#6b9bc3"),
                                   ("leave_site", "leave-site (unseen tower)", "#2C5F8A")]):
    ax.bar(x + (i-1)*w, np.clip(top[col], -0.2, 1), w, label=lab, color=c, zorder=3)
ax.axhline(0, color="#8a8a8a", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(top.model, rotation=30, ha="right", fontsize=7.5)
ax.set_ylabel("$R^2$"); ax.set_ylim(-0.2, 1.0)
ax.legend(frameon=False, fontsize=7, loc="upper right")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.set_title("Predictive skill by validation scheme (13 sites)", fontsize=9.5, fontweight="bold")
fig.savefig(f"{FIG}/model_comparison_runnable.png", dpi=200, bbox_inches="tight")
plt.show()

We consistently find **tree ensembles (ExtraTrees / RandomForest)** give the best
leave-site skill, while the **Gaussian process** wins K-fold interpolation. Deep nets are
omitted here — at n≈833 they don't beat the trees and add instability.

## Feature importance — what actually transfers

Gini importance flatters whatever the trees split on in-sample. The honest measure is
permutation importance on **held-out sites**: shuffle a feature, see how much unseen-tower
skill drops.

In [ ]:
# Feature importance the honest way: permutation importance measured on HELD-OUT
# sites (leave-site-out), averaged over sites. This ranks what actually transfers.
from sklearn.inspection import permutation_importance
perm = np.zeros(len(FEATS)); n = 0
for s in SITES:
    tr, te = d[d.SITE_ID != s], d[d.SITE_ID == s]
    if len(te) < 5: continue
    m = ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1).fit(
        tr[FEATS].values, tr.ET_closed_mm.values)
    pi = permutation_importance(m, te[FEATS].values, te.ET_closed_mm.values, n_repeats=8, random_state=0)
    perm += np.clip(pi.importances_mean, 0, None); n += 1
imp = pd.Series(perm / n, index=FEATS).sort_values()

col = ["#55A868" if f in SAT else "#4C72B0" for f in imp.index]
fig, ax = plt.subplots(figsize=(4.6, 4.2), constrained_layout=True)
ax.barh(np.arange(len(imp)), imp.values, color=col, height=0.72, zorder=3)
ax.set_yticks(np.arange(len(imp))); ax.set_yticklabels(imp.index, fontsize=7.5)
ax.set_xlabel("Permutation importance (leave-site-out $\\Delta R^2$)")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(fc="#55A868", label="satellite"), Patch(fc="#4C72B0", label="meteorology")],
          frameon=False, fontsize=7, loc="lower right")
ax.set_title("What drives transferable ET skill", fontsize=9.5, fontweight="bold")
fig.savefig(f"{FIG}/feature_importance_runnable.png", dpi=200, bbox_inches="tight")
plt.show()
print("Top drivers:", list(imp.sort_values(ascending=False).index[:5]))

Reference ET (`ETo_mm`) dominates, followed by seasonal timing and the water/moisture
indices (NDWI, LST); raw greenness (LAI/NDVI/EVI2/SAVI) barely moves unseen-site skill —
the marshes are too spectrally similar in greenness for it to discriminate.

## 8. Feature-group ablation — how few / which inputs do we need?

Before any formal selection, we just train on progressively fewer (or different) input
groups and score each on leave-site-out. This shows directly how much each group adds.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
GROUPS = {
    "ETo only (1)":         ["ETo_mm"],
    "meteorology (7)":      MET,
    "greenness (4)":        ["LAI", "EVI2", "SAVI", "NDVI"],
    "water + LST (3)":      ["NDWI", "MNDWI", "LST_K"],
    "satellite (7)":        SAT,
    "met + water/LST (10)": MET + ["NDWI", "MNDWI", "LST_K"],
    "FULL (14)":            FEATS,
}
# the top-k most important inputs (ranking from the section-7 permutation importance)
ranked = imp.sort_values(ascending=False).index.tolist()
GROUPS["top-6 (importance)"] = ranked[:6]
GROUPS["top-7 (importance)"] = ranked[:7]
print("top-7 by importance:", ranked[:7])
etm = lambda: ExtraTreesRegressor(400, min_samples_leaf=2, random_state=0, n_jobs=-1)
rows = []
for name, cols in GROUPS.items():
    rs = evaluate(etm, d, "site", feats=cols)[0]
    rows.append((name, len(cols), round(rs, 3)))
    print(f"  {name:<22} n={len(cols):<3} leave-site R2 = {rs:.3f}", flush=True)
abl = pd.DataFrame(rows, columns=["feature set", "n", "leave_site_R2"])
abl

In [ ]:
order = abl.sort_values("leave_site_R2")
col = ["#C44E52" if s in ("greenness (4)", "ETo only (1)") else
       ("#DD8452" if ("satellite" in s or "water" in s) else "#4C72B0") for s in order["feature set"]]
fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.barh(np.arange(len(order)), order.leave_site_R2.clip(lower=-0.05), color=col, zorder=3)
ax.set_yticks(np.arange(len(order))); ax.set_yticklabels(order["feature set"], fontsize=8.5)
ax.axvline(order.leave_site_R2.max(), color="#888", ls="--", lw=0.8)
ax.set_xlabel("leave-site-out $R^2$"); ax.set_xlim(-0.1, 0.8)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.set_title("Fewer inputs, same skill — feature-group ablation", fontsize=10, fontweight="bold")
plt.show()

**Takeaway:** meteorology alone already matches the full 14-feature model; greenness
alone is noise (R² ≈ 0); the only satellite signal that helps is **water + LST**. In fact the
**top-7 inputs by importance — 6 meteorology + NDWI — reach leave-site R² ≈ 0.72, matching the
full model with half the features** (top-6 ≈ 0.72 too). So the model can be trimmed hard with
no loss, which the VIF and AIC/BIC selection below make formal.

## 9. Feature selection I — multicollinearity (VIF)

Several predictors are near-duplicates: the greenness indices are all monotone functions
of NIR-Red, and ETo is a combination of the met variables. We quantify this with the
variance inflation factor (VIF) and prune the redundant features.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

def vif(cols):
    Z = StandardScaler().fit_transform(d[cols].values)
    return pd.Series({f: 1/max(1e-9, 1-LinearRegression().fit(
        np.delete(Z, j, 1), Z[:, j]).score(np.delete(Z, j, 1), Z[:, j]))
        for j, f in enumerate(cols)}).sort_values(ascending=False)

print("VIF (full 14 features):"); print(vif(FEATS).round(1).to_string())

cmat = d[FEATS].corr().abs()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cmat.values, cmap="RdBu_r", vmin=0, vmax=1)
ax.set_xticks(range(len(FEATS))); ax.set_xticklabels(FEATS, rotation=90, fontsize=6.5)
ax.set_yticks(range(len(FEATS))); ax.set_yticklabels(FEATS, fontsize=6.5)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
ax.set_title("|correlation| among predictors", fontsize=10, fontweight="bold")
plt.show()

# iterative elimination: drop the highest VIF until all <= 10
cols = list(FEATS)
while len(cols) > 2 and vif(cols).max() > 10:
    worst = vif(cols).idxmax(); print("drop", worst, " VIF", round(vif(cols).max(), 1)); cols.remove(worst)
SELECTED = cols
print("\nVIF-pruned set (", len(SELECTED), "):", SELECTED)

VIF flags the greenness indices as severe (EVI2/SAVI in the thousands). Eliminating
the worst leaves an ~11-feature decorrelated set (all VIF ≤ ~8).

## 10. Feature selection II — AIC/BIC (linear model)

AIC/BIC are defined for the maximum-likelihood (OLS) model, so we use them for forward
stepwise selection of a linear ET model. They are **not** valid for the tree ensembles (no
likelihood, no well-defined parameter count), so model choice among those stays on
leave-site CV.

In [ ]:
import statsmodels.api as sm
y = d.ET_closed_mm.values
Xs = pd.DataFrame(StandardScaler().fit_transform(d[FEATS].values), columns=FEATS, index=d.index)

def ic(cols):
    m = sm.OLS(y, sm.add_constant(Xs[cols])).fit(); return m.aic, m.bic

def forward(which):
    rem, sel, cur = list(FEATS), [], 1e18
    while rem:
        f, s = min(((f, ic(sel + [f])[0 if which == "aic" else 1]) for f in rem), key=lambda t: t[1])
        if s < cur - 1e-6:
            cur = s; sel.append(f); rem.remove(f)
        else:
            break
    return sel

aic_set, bic_set = forward("aic"), forward("bic")
for nm, cs in [("full(14)", FEATS), ("VIF", SELECTED), ("AIC-step", aic_set), ("BIC-step", bic_set)]:
    a, b = ic(cs); print(f"  {nm:<9} k={len(cs):<3} AIC={a:8.1f}  BIC={b:8.1f}")
print("\nBIC-selected:", bic_set)

BIC (the stricter penalty) drops **every greenness index**, keeping only water
(NDWI) and thermal (LST) among the satellite features — the same verdict as VIF and the
feature-importance ablation.

## 11. Retrain all models on the selected features → pick + save the best

We retrain the whole model zoo on the pruned features and rank by leave-site R² (spatial
transfer). The winner is refit on all the data and saved as the production model.

In [ ]:
import joblib, json
from sklearn.base import clone

rows = []
for name, mk in ZOO.items():
    rows.append((name, round(evaluate(mk, d, "site", feats=SELECTED)[0], 3)))
rank = pd.DataFrame(rows, columns=["model", "leave_site_R2"]).sort_values("leave_site_R2", ascending=False)
print(rank.to_string(index=False))

best = rank.iloc[0]["model"]
prod = clone(ZOO[best]()).fit(d[SELECTED].values, d.ET_closed_mm.values)
meta = {"model": best, "features": SELECTED, "leave_site_R2": float(rank.iloc[0]["leave_site_R2"]),
        "n_train": int(len(d)), "n_sites": int(d.SITE_ID.nunique())}
joblib.dump({**meta, "model": prod}, f"{PROC}/final_model.joblib")  # fitted model wins the key
json.dump(meta, open(f"{PROC}/final_model.json", "w"), indent=2)
print(f"\nPRODUCTION MODEL: {best} on {len(SELECTED)} features  ->  saved final_model.joblib")

**Result:** ExtraTrees on the decorrelated feature set is the production model
(leave-site R² ≈ 0.72). Multicollinearity (VIF), AIC/BIC, and the importance ablation all
converge on the same parsimonious set — **water + thermal + meteorology, greenness dropped**
— and this is the model `04_spatial_prediction` uses to map ET.